In [1]:
from db.conf import create_db_engine, get_async_session
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
engine = create_db_engine()
db = get_async_session(engine)

In [5]:
from datetime import datetime, timedelta
from uuid import UUID

from db.repositories.helpers import full_video_data
from db.repositories.videos import VideoRepository

async with db() as session:
  video_repo = VideoRepository(session)
  videos = await video_repo.get_video_by_topic(
      UUID("6aac9d4a-acd0-11f0-b4c6-27ffcea6845c"),
      load_annotations=True,
      load_meta=True,
      max_videos=10
  )

  video_data = [full_video_data(v) for v in videos]

2025-10-21 11:44:02,572 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-21 11:44:02,573 INFO sqlalchemy.engine.Engine SELECT videos.id, videos.url, videos.source, videos.author_id, videos.uploaded_at, videos.likes, videos.views, videos.comments, videos.revision, videos.extra_data, videos.created_at, videos.updated_at 
FROM videos JOIN video_topics ON videos.id = video_topics.video_id 
WHERE video_topics.topic_id = $1::UUID ORDER BY videos.uploaded_at DESC 
 LIMIT $2::INTEGER
2025-10-21 11:44:02,574 INFO sqlalchemy.engine.Engine [cached since 198.8s ago] (UUID('6aac9d4a-acd0-11f0-b4c6-27ffcea6845c'), 10)
2025-10-21 11:44:02,577 INFO sqlalchemy.engine.Engine SELECT video_meta.video_id AS video_meta_video_id, video_meta.id AS video_meta_id, video_meta.source AS video_meta_source, video_meta.revision AS video_meta_revision, video_meta.value AS video_meta_value, video_meta.value_tsv AS video_meta_value_tsv, video_meta.meta AS video_meta_meta, video_meta.created_at AS video_meta_creat

In [6]:
from core.agents.common import TemplateManager, gemini_2_5_flash_lite, gemini_2_5_flash, gpt_5_nano
from core.agents.challenge import ChallengeGenAgent, ChallengeGenAgentRun


agent = ChallengeGenAgent(gemini_2_5_flash_lite(), TemplateManager())

In [8]:
run_input = ChallengeGenAgentRun(topic="Dota 2 - Live Gameplay", videos=video_data, languages=["en", "fr"])

result = await agent.run(run_input)
result

ChallengeGenAgentResponse(challenges=[Challenge(translations=[ChallengeName(lang='en', name='Find a streamer reacting intensely to MOBA gameplay'), ChallengeName(lang='fr', name='Trouvez un streamer réagissant intensément au gameplay MOBA')], videos=[VideoChallenge(id=UUID('785d186a-accc-11f0-a8a4-631a8d99c0c0'), reason='The video shows the streamer laughing with his hands on his head, indicating an intense reaction to the Dota 2 gameplay.'), VideoChallenge(id=UUID('786e9496-accc-11f0-a8a4-cf54217a30bc'), reason='The streamer is shown with an intense expression during a chaotic fight, and the transcription includes strong language.'), VideoChallenge(id=UUID('78715d70-accc-11f0-a8a4-471b4f41eaa7'), reason='The transcription includes exclamations like "Take that!" and "Handehoch, bitch!" during the gameplay.'), VideoChallenge(id=UUID('787415d8-accc-11f0-a8a4-132fad0d86c7'), reason='The streamer displays a surprised expression during the MOBA match.'), VideoChallenge(id=UUID('7876de30-acc